In [1]:
!pip -q install duckdb pyarrow pandas numpy tqdm sentence-transformers psutil

In [2]:
import os
import re
import gc
import json
import time
import glob
import math
import shutil
import sqlite3
import numpy as np
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [11]:
BASE_DIR = "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output"
INPUT_PATH = os.path.join(BASE_DIR, "san_pham.parquet")

WORK_DIR = "/content/semantic_search_full_mpnet_bm25_hybrid_eval"
EXPORT_DIR = os.path.join(BASE_DIR, "semantic_search_full_mpnet_bm25_hybrid_eval")
OLD_EXPORT_DIR = os.path.join(BASE_DIR, "semantic_search_full_mpnet_cosine_eval")

CLEAN_PARQUET_PATH = os.path.join(WORK_DIR, "clean_products.parquet")
EMBEDDING_DIR = os.path.join(WORK_DIR, "embedding_shards")
METADATA_DIR = os.path.join(WORK_DIR, "metadata_shards")
METADATA_DB_PATH = os.path.join(WORK_DIR, "metadata.duckdb")
BM25_DB_PATH = os.path.join(WORK_DIR, "bm25_fts.sqlite")
ETL_STATS_PATH = os.path.join(WORK_DIR, "etl_stats.json")
RUN_STATS_PATH = os.path.join(WORK_DIR, "run_stats.json")
BM25_STATS_PATH = os.path.join(WORK_DIR, "bm25_stats.json")
EVAL_METRICS_PATH = os.path.join(WORK_DIR, "category_evaluation_metrics.csv")
EVAL_SUMMARY_PATH = os.path.join(WORK_DIR, "category_evaluation_summary.csv")

MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

MAX_PRODUCTS = None
EMBED_BATCH_ROWS = 30000
ENCODE_BATCH_SIZE = 256
MAX_SEQ_LENGTH = 128

EVAL_QUERY_COUNT = 30
EVAL_K = 10
HYBRID_DENSE_POOL = 60
HYBRID_BM25_POOL = 60
HYBRID_DENSE_WEIGHT = 0.6
HYBRID_BM25_WEIGHT = 0.4

RESET_WORK_DIR = False
LOAD_EXISTING_EXPORT = True
LOAD_COMPATIBLE_OLD_EXPORT = True

FORCE_REBUILD_CLEAN = False
FORCE_REBUILD_EMBEDDINGS = False
FORCE_REBUILD_METADATA_DB = False
FORCE_REBUILD_BM25 = False
FORCE_REBUILD_EVAL = False

SAVE_TO_DRIVE = True
USE_FP16_ON_GPU = True
USE_GPU_FOR_BATCH_RETRIEVAL = True

if LOAD_EXISTING_EXPORT and os.path.exists(EXPORT_DIR) and not os.path.exists(WORK_DIR):
    shutil.copytree(EXPORT_DIR, WORK_DIR)

if LOAD_COMPATIBLE_OLD_EXPORT and (not os.path.exists(EXPORT_DIR)) and os.path.exists(OLD_EXPORT_DIR) and not os.path.exists(WORK_DIR):
    os.makedirs(WORK_DIR, exist_ok=True)
    for name in ["clean_products.parquet", "embedding_shards", "metadata_shards", "metadata.duckdb", "etl_stats.json", "run_stats.json"]:
        src = os.path.join(OLD_EXPORT_DIR, name)
        dst = os.path.join(WORK_DIR, name)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif os.path.exists(src):
            shutil.copy2(src, dst)

if RESET_WORK_DIR and os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(WORK_DIR)
print(EXPORT_DIR)


/content/semantic_search_full_mpnet_bm25_hybrid_eval
/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_full_mpnet_bm25_hybrid_eval


In [4]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def sql_literal(x):
    return "'" + str(x).replace("'", "''") + "'"

def qident(x):
    return '"' + str(x).replace('"', '""') + '"'

def duck_parquet_source(path):
    if os.path.isdir(path):
        return os.path.join(path, "**", "*.parquet")
    return path

def pick_column(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for cand in candidates:
        for c in columns:
            if cand.lower() in c.lower():
                return c
    return None

def duck_text_expr(col, max_len=1200):
    if col is None:
        return "''"
    x = f"CAST({qident(col)} AS VARCHAR)"
    x = f"COALESCE({x}, '')"
    x = f"REGEXP_REPLACE({x}, '\\s+', ' ', 'g')"
    return f"SUBSTR({x}, 1, {int(max_len)})"

def normalize_key_expr(expr):
    x = f"LOWER(COALESCE({expr}, ''))"
    x = f"REGEXP_REPLACE({x}, '[^a-z0-9]+', ' ', 'g')"
    x = f"REGEXP_REPLACE({x}, '\\s+', ' ', 'g')"
    return f"TRIM({x})"

def minmax_norm(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return x
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx - mn < 1e-9:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def dcg_at_k(rels):
    rels = np.asarray(rels, dtype=np.float32)
    if rels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels.size + 2))
    return float(np.sum(rels / discounts))

def ndcg_at_k(rels):
    rels = list(rels)
    ideal = sorted(rels, reverse=True)
    ideal_dcg = dcg_at_k(ideal)
    if ideal_dcg <= 0:
        return 0.0
    return dcg_at_k(rels) / ideal_dcg

def mrr_at_k(rels):
    for i, r in enumerate(rels, start=1):
        if r > 0:
            return 1.0 / i
    return 0.0

def tokenize_for_bm25(text, max_terms=16):
    toks = re.findall(r"[0-9A-Za-zÀ-ỹ]+", str(text).lower())
    keep = []
    seen = set()
    for t in toks:
        if len(t) <= 1:
            continue
        if t not in seen:
            keep.append(t)
            seen.add(t)
        if len(keep) >= max_terms:
            break
    return keep

def make_fts_query(text):
    toks = tokenize_for_bm25(text)
    if not toks:
        return ""
    return " OR ".join([f'"{t}"' for t in toks])

def sorted_files(path, pattern):
    return sorted(glob.glob(os.path.join(path, pattern)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [5]:
etl_start = time.time()

if os.path.exists(CLEAN_PARQUET_PATH) and os.path.exists(ETL_STATS_PATH) and not FORCE_REBUILD_CLEAN:
    with open(ETL_STATS_PATH, "r", encoding="utf-8") as f:
        etl_stats = json.load(f)
    print(json.dumps(etl_stats, ensure_ascii=False, indent=2))
else:
    if os.path.exists(CLEAN_PARQUET_PATH):
        if os.path.isdir(CLEAN_PARQUET_PATH):
            shutil.rmtree(CLEAN_PARQUET_PATH)
        else:
            os.remove(CLEAN_PARQUET_PATH)

    con = duckdb.connect()
    source = duck_parquet_source(INPUT_PATH)
    cols_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet({sql_literal(source)})").fetchdf()
    columns = cols_df["column_name"].tolist()

    asin_col = pick_column(columns, ["asin", "parent_asin", "product_id", "item_id"])
    title_col = pick_column(columns, ["title", "name", "product_name"])
    brand_col = pick_column(columns, ["brand", "store"])
    category_col = pick_column(columns, ["main_category", "category", "categories", "bc"])
    description_col = pick_column(columns, ["description", "desc"])
    features_col = pick_column(columns, ["features", "feature"])
    details_col = pick_column(columns, ["details", "specs", "specifications"])
    price_col = pick_column(columns, ["price", "final_price"])

    asin_expr = duck_text_expr(asin_col, 120)
    title_expr = duck_text_expr(title_col, 400)
    brand_expr = duck_text_expr(brand_col, 200)
    category_expr = duck_text_expr(category_col, 700)
    description_expr = duck_text_expr(description_col, 1500)
    features_expr = duck_text_expr(features_col, 1500)
    details_expr = duck_text_expr(details_col, 1500)
    price_expr = duck_text_expr(price_col, 100)

    limit_clause = "" if MAX_PRODUCTS is None else f"LIMIT {int(MAX_PRODUCTS)}"

    con.execute(f"""
    CREATE OR REPLACE TABLE clean_products AS
    SELECT
        ROW_NUMBER() OVER () - 1 AS row_id,
        asin,
        title,
        brand,
        category,
        price,
        retrieval_text,
        category_key
    FROM (
        SELECT
            {asin_expr} AS asin,
            {title_expr} AS title,
            {brand_expr} AS brand,
            {category_expr} AS category,
            {price_expr} AS price,
            LOWER(REGEXP_REPLACE(CONCAT_WS(' ', {title_expr}, {brand_expr}, {category_expr}, {description_expr}, {features_expr}, {details_expr}), '\\s+', ' ', 'g')) AS retrieval_text,
            {normalize_key_expr(category_expr)} AS category_key
        FROM read_parquet({sql_literal(source)})
    )
    WHERE asin <> ''
      AND title <> ''
      AND LENGTH(retrieval_text) > 10
      AND category_key <> ''
    {limit_clause}
    """)

    clean_count = con.execute("SELECT COUNT(*) FROM clean_products").fetchone()[0]
    raw_count = con.execute(f"SELECT COUNT(*) FROM read_parquet({sql_literal(source)})").fetchone()[0]

    con.execute(f"COPY clean_products TO {sql_literal(CLEAN_PARQUET_PATH)} (FORMAT PARQUET, COMPRESSION 'SNAPPY')")
    con.close()

    etl_stats = {
        "input_path": INPUT_PATH,
        "clean_parquet_path": CLEAN_PARQUET_PATH,
        "raw_count": int(raw_count),
        "clean_count": int(clean_count),
        "asin_col": asin_col,
        "title_col": title_col,
        "brand_col": brand_col,
        "category_col": category_col,
        "description_col": description_col,
        "features_col": features_col,
        "details_col": details_col,
        "price_col": price_col,
        "elapsed_seconds": round(time.time() - etl_start, 2)
    }
    save_json(etl_stats, ETL_STATS_PATH)
    print(json.dumps(etl_stats, ensure_ascii=False, indent=2))


{
  "skipped": false,
  "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
  "clean_parquet_path": "/content/semantic_search_full_mpnet_cosine_eval/clean_products.parquet",
  "raw_count": 2410915,
  "clean_count": 2410765,
  "column_map": {
    "asin": "asin",
    "title": "title",
    "brand": "store",
    "category": "main_category",
    "description": "description",
    "features": "features",
    "details": "details",
    "price": "price"
  },
  "elapsed_seconds": 519.33
}


In [6]:
embedding_start = time.time()
existing_embeddings = sorted_files(EMBEDDING_DIR, "embeddings_*.npy")

if existing_embeddings and os.path.exists(RUN_STATS_PATH) and not FORCE_REBUILD_EMBEDDINGS:
    with open(RUN_STATS_PATH, "r", encoding="utf-8") as f:
        run_stats = json.load(f)
    print(json.dumps(run_stats, ensure_ascii=False, indent=2))
else:
    if os.path.exists(EMBEDDING_DIR):
        shutil.rmtree(EMBEDDING_DIR)
    if os.path.exists(METADATA_DIR):
        shutil.rmtree(METADATA_DIR)
    os.makedirs(EMBEDDING_DIR, exist_ok=True)
    os.makedirs(METADATA_DIR, exist_ok=True)

    model = SentenceTransformer(MODEL_NAME, device=str(device))
    model.max_seq_length = MAX_SEQ_LENGTH
    if device.type == "cuda" and USE_FP16_ON_GPU:
        model = model.half()

    dataset = ds.dataset(CLEAN_PARQUET_PATH, format="parquet")
    scanner = dataset.scanner(
        columns=["row_id", "asin", "title", "brand", "category", "price", "category_key", "retrieval_text"],
        batch_size=EMBED_BATCH_ROWS
    )

    row_count = 0
    shard_id = 0
    embedding_dim = None

    for batch in tqdm(scanner.to_batches(), desc="Embedding shards"):
        df = batch.to_pandas()
        if len(df) == 0:
            continue

        texts = df["retrieval_text"].fillna("").astype(str).tolist()
        embeddings = model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        if embedding_dim is None:
            embedding_dim = int(embeddings.shape[1])

        emb_path = os.path.join(EMBEDDING_DIR, f"embeddings_{shard_id:06d}.npy")
        meta_path = os.path.join(METADATA_DIR, f"metadata_{shard_id:06d}.parquet")

        np.save(emb_path, embeddings)
        meta = df[["row_id", "asin", "title", "brand", "category", "price", "category_key"]].copy()
        pq.write_table(pa.Table.from_pandas(meta, preserve_index=False), meta_path, compression="snappy")

        row_count += len(df)
        shard_id += 1

    run_stats = {
        "model_name": MODEL_NAME,
        "n_products_embedded": int(row_count),
        "n_shards": int(shard_id),
        "embedding_dim": int(embedding_dim) if embedding_dim is not None else None,
        "encode_batch_size": int(ENCODE_BATCH_SIZE),
        "max_seq_length": int(MAX_SEQ_LENGTH),
        "elapsed_seconds": round(time.time() - embedding_start, 2)
    }
    save_json(run_stats, RUN_STATS_PATH)
    print(json.dumps(run_stats, ensure_ascii=False, indent=2))


{
  "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
  "work_dir": "/content/semantic_search_full_mpnet_cosine_eval",
  "export_dir": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_full_mpnet_cosine_eval",
  "model_name": "paraphrase-multilingual-mpnet-base-v2",
  "retrieval_method": "exact_cosine_similarity_same_as_demo",
  "etl": {
    "skipped": false,
    "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
    "clean_parquet_path": "/content/semantic_search_full_mpnet_cosine_eval/clean_products.parquet",
    "raw_count": 2410915,
    "clean_count": 2410765,
    "column_map": {
      "asin": "asin",
      "title": "title",
      "brand": "store",
      "category": "main_category",
      "description": "description",
      "features": "features",
      "details": "details",
      "price": "price"
    },
    "elap

In [7]:
metadata_start = time.time()

if os.path.exists(METADATA_DB_PATH) and not FORCE_REBUILD_METADATA_DB:
    con = duckdb.connect(METADATA_DB_PATH, read_only=True)
    n_meta = con.execute("SELECT COUNT(*) FROM products").fetchone()[0]
    con.close()
    print({"metadata_db_path": METADATA_DB_PATH, "n_products": int(n_meta)})
else:
    if os.path.exists(METADATA_DB_PATH):
        os.remove(METADATA_DB_PATH)

    con = duckdb.connect(METADATA_DB_PATH)
    con.execute(f"""
    CREATE TABLE products AS
    SELECT *
    FROM read_parquet({sql_literal(os.path.join(METADATA_DIR, "*.parquet"))})
    """)
    con.execute("CREATE INDEX IF NOT EXISTS idx_products_row_id ON products(row_id)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_products_category_key ON products(category_key)")
    n_meta = con.execute("SELECT COUNT(*) FROM products").fetchone()[0]
    con.close()

    print({
        "metadata_db_path": METADATA_DB_PATH,
        "n_products": int(n_meta),
        "elapsed_seconds": round(time.time() - metadata_start, 2)
    })


{'metadata_db_path': '/content/semantic_search_full_mpnet_bm25_hybrid_eval/metadata.duckdb', 'n_products': 2410765}


In [8]:
bm25_start = time.time()

if os.path.exists(BM25_DB_PATH) and os.path.exists(BM25_STATS_PATH) and not FORCE_REBUILD_BM25:
    with open(BM25_STATS_PATH, "r", encoding="utf-8") as f:
        bm25_stats = json.load(f)
    print(json.dumps(bm25_stats, ensure_ascii=False, indent=2))
else:
    if os.path.exists(BM25_DB_PATH):
        os.remove(BM25_DB_PATH)

    conn = sqlite3.connect(BM25_DB_PATH)
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode = WAL")
    cur.execute("PRAGMA synchronous = OFF")
    cur.execute("PRAGMA temp_store = MEMORY")
    cur.execute("CREATE VIRTUAL TABLE products_fts USING fts5(row_id UNINDEXED, retrieval_text, tokenize='unicode61')")

    dataset = ds.dataset(CLEAN_PARQUET_PATH, format="parquet")
    scanner = dataset.scanner(columns=["row_id", "retrieval_text"], batch_size=50000)

    total = 0
    for batch in tqdm(scanner.to_batches(), desc="Building BM25 FTS"):
        df = batch.to_pandas()
        rows = [(int(r.row_id), str(r.retrieval_text)) for r in df.itertuples(index=False)]
        cur.executemany("INSERT INTO products_fts(row_id, retrieval_text) VALUES (?, ?)", rows)
        total += len(rows)
        if total % 300000 == 0:
            conn.commit()

    conn.commit()
    cur.execute("INSERT INTO products_fts(products_fts) VALUES('optimize')")
    conn.commit()
    conn.close()

    bm25_stats = {
        "bm25_db_path": BM25_DB_PATH,
        "n_indexed": int(total),
        "engine": "SQLite FTS5 BM25",
        "elapsed_seconds": round(time.time() - bm25_start, 2)
    }
    save_json(bm25_stats, BM25_STATS_PATH)
    print(json.dumps(bm25_stats, ensure_ascii=False, indent=2))


Building BM25 FTS: 0it [00:00, ?it/s]

{
  "bm25_db_path": "/content/semantic_search_full_mpnet_bm25_hybrid_eval/bm25_fts.sqlite",
  "n_indexed": 2410765,
  "engine": "SQLite FTS5 BM25",
  "elapsed_seconds": 561.65
}


In [9]:
retrieval_model = SentenceTransformer(MODEL_NAME, device=str(device))
retrieval_model.max_seq_length = MAX_SEQ_LENGTH
if device.type == "cuda" and USE_FP16_ON_GPU:
    retrieval_model = retrieval_model.half()

embedding_files = sorted_files(EMBEDDING_DIR, "embeddings_*.npy")
metadata_files = sorted_files(METADATA_DIR, "metadata_*.parquet")

def fetch_metadata(row_ids):
    if not row_ids:
        return pd.DataFrame(columns=["row_id", "asin", "title", "brand", "category", "price", "category_key"])
    con = duckdb.connect(METADATA_DB_PATH, read_only=True)
    ids = ",".join(str(int(x)) for x in row_ids)
    df = con.execute(f"""
        SELECT row_id, asin, title, brand, category, price, category_key
        FROM products
        WHERE row_id IN ({ids})
    """).fetchdf()
    con.close()
    return df

def search_bm25(query, top_k=10):
    fts = make_fts_query(query)
    if not fts:
        return []
    conn = sqlite3.connect(BM25_DB_PATH)
    cur = conn.cursor()
    try:
        rows = cur.execute(
            "SELECT row_id, bm25(products_fts) AS score FROM products_fts WHERE products_fts MATCH ? ORDER BY score LIMIT ?",
            (fts, int(top_k))
        ).fetchall()
    except Exception:
        rows = []
    conn.close()
    return [(int(row_id), float(-score)) for row_id, score in rows]

def search_dense(query, top_k=10):
    q = retrieval_model.encode(
        [query],
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    best_scores = []
    best_ids = []

    if device.type == "cuda" and USE_GPU_FOR_BATCH_RETRIEVAL:
        q_t = torch.from_numpy(q).to(device)
        for emb_path, meta_path in zip(embedding_files, metadata_files):
            arr = np.load(emb_path, mmap_mode="r")
            meta = pq.read_table(meta_path, columns=["row_id"]).to_pandas()["row_id"].to_numpy()
            mat = torch.from_numpy(np.asarray(arr)).to(device)
            scores = torch.mv(mat, q_t).detach().cpu().numpy()
            local_k = min(top_k, len(scores))
            idx = np.argpartition(scores, -local_k)[-local_k:]
            best_scores.extend(scores[idx].tolist())
            best_ids.extend(meta[idx].astype(int).tolist())
            del mat
            torch.cuda.empty_cache()
    else:
        for emb_path, meta_path in zip(embedding_files, metadata_files):
            arr = np.load(emb_path, mmap_mode="r")
            meta = pq.read_table(meta_path, columns=["row_id"]).to_pandas()["row_id"].to_numpy()
            scores = np.asarray(arr).dot(q)
            local_k = min(top_k, len(scores))
            idx = np.argpartition(scores, -local_k)[-local_k:]
            best_scores.extend(scores[idx].tolist())
            best_ids.extend(meta[idx].astype(int).tolist())

    if not best_scores:
        return []

    best_scores = np.asarray(best_scores, dtype=np.float32)
    best_ids = np.asarray(best_ids, dtype=np.int64)
    k = min(top_k, len(best_scores))
    idx = np.argsort(best_scores)[::-1][:k]
    return [(int(best_ids[i]), float(best_scores[i])) for i in idx]

def search_hybrid(query, top_k=10):
    dense = search_dense(query, HYBRID_DENSE_POOL)
    bm25 = search_bm25(query, HYBRID_BM25_POOL)

    dense_ids = [x[0] for x in dense]
    bm25_ids = [x[0] for x in bm25]
    all_ids = list(dict.fromkeys(dense_ids + bm25_ids))

    dense_map = {rid: score for rid, score in dense}
    bm25_map = {rid: score for rid, score in bm25}

    dense_norm_values = minmax_norm([dense_map.get(rid, 0.0) for rid in all_ids])
    bm25_norm_values = minmax_norm([bm25_map.get(rid, 0.0) for rid in all_ids])

    scored = []
    for i, rid in enumerate(all_ids):
        score = HYBRID_DENSE_WEIGHT * float(dense_norm_values[i]) + HYBRID_BM25_WEIGHT * float(bm25_norm_values[i])
        scored.append((int(rid), score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

def search_products(query, method="HYBRID", top_k=10):
    if method.upper() == "BM25":
        return search_bm25(query, top_k)
    if method.upper() == "SBERT":
        return search_dense(query, top_k)
    return search_hybrid(query, top_k)

for method in ["BM25", "SBERT", "HYBRID"]:
    t0 = time.time()
    results = search_products("laptop học lập trình", method=method, top_k=5)
    meta = fetch_metadata([r[0] for r in results])
    print(method, "latency", round(time.time() - t0, 3), "seconds")
    display(meta[["row_id", "title", "category"]].head(5))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

/tmp/ipykernel_692/954343956.py:55: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  mat = torch.from_numpy(np.asarray(arr)).to(device)


SBERT latency 31.214 seconds


,row_id,title,category
0,134588,"Lenovo ThinkPad 11e 11.6"" LED Chromebook Lapto...",Computers
1,490030,"Lenovo ThinkPad 11E (5th Gen) 11.6"" HD Busines...",Computers
2,1085958,Lenovo - 300W Gen 3-2-in-1 Educational Compute...,Computers
3,1310558,Dell Chromebook 11 11.6' Notebook - Intel Cele...,Computers
4,1983752,"Lenovo ThinkPad 11e Laptop 11.6"", Intel Celero...",Computers


HYBRID latency 34.205 seconds


,row_id,title,category
0,134588,"Lenovo ThinkPad 11e 11.6"" LED Chromebook Lapto...",Computers
1,490030,"Lenovo ThinkPad 11E (5th Gen) 11.6"" HD Busines...",Computers
2,1085958,Lenovo - 300W Gen 3-2-in-1 Educational Compute...,Computers
3,1310558,Dell Chromebook 11 11.6' Notebook - Intel Cele...,Computers
4,1983752,"Lenovo ThinkPad 11e Laptop 11.6"", Intel Celero...",Computers


In [12]:
eval_start = time.time()

if os.path.exists(EVAL_SUMMARY_PATH) and os.path.exists(EVAL_METRICS_PATH) and not FORCE_REBUILD_EVAL:
    summary_df = pd.read_csv(EVAL_SUMMARY_PATH)
    metrics_df = pd.read_csv(EVAL_METRICS_PATH)
    display(summary_df)
else:
    con = duckdb.connect(METADATA_DB_PATH, read_only=True)
    eval_queries = con.execute(f"""
        SELECT row_id, title, category_key
        FROM products
        WHERE title IS NOT NULL
          AND title <> ''
          AND category_key IS NOT NULL
          AND category_key <> ''
        USING SAMPLE {int(EVAL_QUERY_COUNT)} ROWS
    """).fetchdf()
    con.close()

    rows = []
    methods = ["BM25", "SBERT", "HYBRID"]

    for q in tqdm(eval_queries.itertuples(index=False), total=len(eval_queries), desc="Category evaluation"):
        query_row_id = int(q.row_id)
        query_text = str(q.title)
        truth_category = str(q.category_key)

        for method in methods:
            t0 = time.time()
            results = search_products(query_text, method=method, top_k=EVAL_K + 5)
            results = [(rid, score) for rid, score in results if int(rid) != query_row_id][:EVAL_K]
            latency = time.time() - t0

            row_ids = [rid for rid, _ in results]
            meta = fetch_metadata(row_ids)
            cat_map = {int(r.row_id): str(r.category_key) for r in meta.itertuples(index=False)}

            rels = [1 if cat_map.get(int(rid), "") == truth_category else 0 for rid, _ in results]
            while len(rels) < EVAL_K:
                rels.append(0)

            rows.append({
                "query_row_id": query_row_id,
                "query_title": query_text,
                "truth_category_key": truth_category,
                "method": method,
                "precision_at_10": float(np.mean(rels[:EVAL_K])),
                "hit_at_10": float(1 if np.sum(rels[:EVAL_K]) > 0 else 0),
                "mrr_at_10": float(mrr_at_k(rels[:EVAL_K])),
                "ndcg_at_10": float(ndcg_at_k(rels[:EVAL_K])),
                "latency_seconds": float(latency)
            })

    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv(EVAL_METRICS_PATH, index=False)

    summary_df = metrics_df.groupby("method", as_index=False).agg(
        n_eval_queries=("query_row_id", "count"),
        mean_precision_at_10=("precision_at_10", "mean"),
        mean_hit_at_10=("hit_at_10", "mean"),
        mean_mrr_at_10=("mrr_at_10", "mean"),
        mean_ndcg_at_10=("ndcg_at_10", "mean"),
        mean_latency_seconds=("latency_seconds", "mean")
    )
    summary_df.to_csv(EVAL_SUMMARY_PATH, index=False)

    display(summary_df)
    print("elapsed_seconds", round(time.time() - eval_start, 2))


Category evaluation:   0%|          | 0/30 [00:00<?, ?it/s]

,method,n_eval_queries,mean_precision_at_10,mean_hit_at_10,mean_mrr_at_10,mean_ndcg_at_10,mean_latency_seconds
0,BM25,30,0.726667,0.933333,0.785317,0.822018,6.386088
1,HYBRID,30,0.713333,0.900000,0.772354,0.790171,41.927255
2,SBERT,30,0.720000,0.900000,0.829167,0.826575,30.435145


elapsed_seconds 2383.99


In [13]:
export_start = time.time()

if SAVE_TO_DRIVE:
    os.makedirs(EXPORT_DIR, exist_ok=True)

    for name in [
        "clean_products.parquet",
        "metadata.duckdb",
        "bm25_fts.sqlite",
        "etl_stats.json",
        "run_stats.json",
        "bm25_stats.json",
        "category_evaluation_metrics.csv",
        "category_evaluation_summary.csv"
    ]:
        src = os.path.join(WORK_DIR, name)
        dst = os.path.join(EXPORT_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, dst)

    for dirname in ["embedding_shards", "metadata_shards"]:
        src_dir = os.path.join(WORK_DIR, dirname)
        dst_dir = os.path.join(EXPORT_DIR, dirname)
        os.makedirs(dst_dir, exist_ok=True)
        for src in glob.glob(os.path.join(src_dir, "*")):
            dst = os.path.join(dst_dir, os.path.basename(src))
            if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
                shutil.copy2(src, dst)

print({
    "export_dir": EXPORT_DIR,
    "elapsed_seconds": round(time.time() - export_start, 2)
})


{'export_dir': '/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_full_mpnet_bm25_hybrid_eval', 'elapsed_seconds': 177.85}


In [14]:
print("Artifacts")
for path in [
    CLEAN_PARQUET_PATH,
    METADATA_DB_PATH,
    BM25_DB_PATH,
    ETL_STATS_PATH,
    RUN_STATS_PATH,
    BM25_STATS_PATH,
    EVAL_METRICS_PATH,
    EVAL_SUMMARY_PATH
]:
    print(path, os.path.exists(path))

print("embedding_shards", len(sorted_files(EMBEDDING_DIR, "embeddings_*.npy")))
print("metadata_shards", len(sorted_files(METADATA_DIR, "metadata_*.parquet")))

if os.path.exists(EVAL_SUMMARY_PATH):
    display(pd.read_csv(EVAL_SUMMARY_PATH))


Artifacts
/content/semantic_search_full_mpnet_bm25_hybrid_eval/clean_products.parquet True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/metadata.duckdb True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/bm25_fts.sqlite True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/etl_stats.json True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/run_stats.json True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/bm25_stats.json True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/category_evaluation_metrics.csv True
/content/semantic_search_full_mpnet_bm25_hybrid_eval/category_evaluation_summary.csv True
embedding_shards 97
metadata_shards 97


,method,n_eval_queries,mean_precision_at_10,mean_hit_at_10,mean_mrr_at_10,mean_ndcg_at_10,mean_latency_seconds
0,BM25,30,0.726667,0.933333,0.785317,0.822018,6.386088
1,HYBRID,30,0.713333,0.900000,0.772354,0.790171,41.927255
2,SBERT,30,0.720000,0.900000,0.829167,0.826575,30.435145
